# Conway's Game of Life

Modelo em Z3 com Bounded Model Checking de um automato celular com grelha N×N, celulas iniciais aleatorias (constantes) e celulas finais mortas.

- Transicao - o parametro 'MODE' escolhe entre 'SYNC' (transicoes de celulas sincronizadas) e 'ASYNC' (so uma celula normal transita por passo).
- P1 - nunca se alcanca o estado de erro (todas as celulas normais mortas).
- P2 - nenhuma celula normal esta permanentemente morta ou permanentemente viva.

Ambas as propriedades sao falsas nos dois modos.

## 1. Parametros

In [ ]:
from z3 import Bool, BoolVal, And, Or, Not, If, PbEq, Solver, sat, is_true

MODE = "SYNC"     # SYNC: todas as celulas transitam em simultaneo | ASYNC: uma so celula por passo
N = 17            # N > 16
K_MAX = 12        # profundidade maxima do BMC

assert MODE in ("SYNC", "ASYNC") and N > 16

## 2. Tipos de celulas

As contagens 2N+1 e 2N+3 so sao consistentes se (0,N+1) e (N+1,0) forem finais (assumido ser verdadeiro).

In [5]:
def is_normal(i, j):
    return 1 <= i <= N and 1 <= j <= N

def is_initial(i, j):
    return (i == 0 and 0 <= j <= N) or (j == 0 and 1 <= i <= N)

def is_final(i, j):
    on_border = i in (0, N + 1) or j in (0, N + 1)
    return on_border and not is_initial(i, j)

NORMAL  = [(i, j) for i in range(1, N + 1) for j in range(1, N + 1)]
INITIAL = [(i, j) for i in range(N + 2) for j in range(N + 2) if is_initial(i, j)]
FINAL   = [(i, j) for i in range(N + 2) for j in range(N + 2) if is_final(i, j)]
assert len(NORMAL) == N * N and len(INITIAL) == 2 * N + 1 and len(FINAL) == 2 * N + 3
print(len(NORMAL), len(INITIAL), len(FINAL))

289 35 37


## 3. Variaveis de estado

In [6]:
_X, _S = {}, {}

def X(t, i, j):
    """celula normal (i,j) no instante t"""
    if (t, i, j) not in _X:
        _X[(t, i, j)] = Bool(f"x_{t}_{i}_{j}")
    return _X[(t, i, j)]

def SEL(t, i, j):
    """(ASYNC) a celula (i,j) e a que transita no passo t -> t+1"""
    if (t, i, j) not in _S:
        _S[(t, i, j)] = Bool(f"sel_{t}_{i}_{j}")
    return _S[(t, i, j)]

# celulas iniciais: valores livres mas constantes ao longo do tempo
B = {c: Bool(f"b_{c[0]}_{c[1]}") for c in INITIAL}

def cell(t, i, j):
    if is_normal(i, j):
        return X(t, i, j)
    if is_initial(i, j):
        return B[(i, j)]
    return BoolVal(False)          # celula final

## 4. A FSM: Init, Trans e predicados

- rule aplica B3/S23 com vizinhanca de Moore, as celulas finais valem 0 e sao omitidas da contagem.
- trans depende de MODE.
- error(t) - todas as celulas normais mortas.
- stable(t) - a regra nao altera nenhuma celula normal (ponto fixo, valido nos dois modos).

In [7]:
def init():
    return And([X(0, i, j) if (i <= 3 and j <= 3) else Not(X(0, i, j)) for (i, j) in NORMAL])

def rule(t, i, j):
    nb = [(a, b) for a in (i - 1, i, i + 1) for b in (j - 1, j, j + 1)
          if (a, b) != (i, j) and not is_final(a, b)]
    pb = [(cell(t, a, b), 1) for (a, b) in nb]
    return Or(PbEq(pb, 3), And(X(t, i, j), PbEq(pb, 2)))

def trans(t):
    cs = []
    if MODE == "SYNC":
        for (i, j) in NORMAL:
            cs.append(X(t + 1, i, j) == rule(t, i, j))
    else:   # ASYNC: exatamente uma celula e escolhida; so essa aplica a regra
        cs.append(PbEq([(SEL(t, i, j), 1) for (i, j) in NORMAL], 1))
        for (i, j) in NORMAL:
            cs.append(X(t + 1, i, j) == If(SEL(t, i, j), rule(t, i, j), X(t, i, j)))
    return And(cs)

def error(t):
    return Not(Or([X(t, i, j) for (i, j) in NORMAL]))

def stable(t):
    return And([X(t, i, j) == rule(t, i, j) for (i, j) in NORMAL])

## 5. BMC

In [8]:
def bmc(goal, kmax):
    """menor k <= kmax tal que Init ∧ Trans^k ∧ goal(k) e satisfazivel"""
    s = Solver()
    s.add(init())
    for k in range(kmax + 1):
        s.push()
        s.add(goal(k))
        if s.check() == sat:
            return k, s.model()
        s.pop()
        s.add(trans(k))
    return None, None

## 6. Apresentacao e validacao cruzada

replay repete o traco com um simulador independente e compara com o modelo do Z3.

In [9]:
def val(m, e):
    return is_true(m.eval(e, model_completion=True))

def show(m, t):
    print(f"  t={t}")
    for i in range(1, N + 1):
        print("   " + "".join("#" if val(m, X(t, i, j)) else "." for j in range(1, N + 1)))

def replay(m, k):
    b = {c: val(m, B[c]) for c in INITIAL}
    g = {(i, j): False for i in range(N + 2) for j in range(N + 2)}
    g.update(b)
    for (i, j) in NORMAL:
        g[(i, j)] = (i <= 3 and j <= 3)

    def nxt(i, j):
        n = sum(g[(a, c)] for a in (i - 1, i, i + 1) for c in (j - 1, j, j + 1) if (a, c) != (i, j))
        return n == 3 or (g[(i, j)] and n == 2)

    for t in range(k):
        if MODE == "SYNC":
            new = {c: nxt(*c) for c in NORMAL}
        else:
            chosen = [c for c in NORMAL if val(m, SEL(t, *c))]
            assert len(chosen) == 1
            new = {chosen[0]: nxt(*chosen[0])}
        g.update(new)
    return all(g[c] == val(m, X(k, *c)) for c in NORMAL)

def report(k, m, show_grids=True):
    if k is None:
        print(f"  Nenhum contra-exemplo com k <= {K_MAX}.")
        return
    print(f"  CONTRA-EXEMPLO com k = {k} passos "
          f"(validacao cruzada: {'OK' if replay(m, k) else 'FALHOU'})")
    print("  celulas iniciais vivas:", [c for c in INITIAL if val(m, B[c])])
    if MODE == "ASYNC":
        for t in range(k):
            c = [c for c in NORMAL if val(m, SEL(t, *c))][0]
            print(f"  passo {t}->{t+1}: transita a celula {c}")
    if show_grids:
        for t in range(k + 1):
            show(m, t)

## 7. P1 - "Nunca se alcanca o estado de erro"

Procura 'Init && Trans^k && erro(k)' . Satisfazivel -> contra-exemplo.

In [10]:
print(f"N={N}  modo={MODE}  K_MAX={K_MAX}")
k1, m1 = bmc(error, K_MAX)
report(k1, m1)

N=17  modo=SYNC  K_MAX=12
  CONTRA-EXEMPLO com k = 3 passos (validacao cruzada: OK)
  celulas iniciais vivas: [(0, 0), (0, 1), (0, 2), (0, 8), (0, 11), (0, 12), (2, 0)]
  t=0
   ###..............
   ###..............
   ###..............
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
  t=1
   .................
   ...#.............
   ..#..............
   .#...............
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
  t=2
   .................
   .................
   ..#..............
   .................
   ............

## 8. P2 - "Nenhuma celula normal e permanentemente morta ou viva"

Le-se como: em toda a execucao, cada celula normal muda de valor infinitas vezes.

Um contra-exemplo e um estado alcancavel e estavel (nenhuma transicao o altera): a partir dele todas as celulas ficam constantes para sempre, para qualquer escalonamento e mesmo com fairness. (Sem este criterio, no modo `ASYNC` um escalonador que nunca escolhe uma celula violaria a propriedade.)

In [11]:
print(f"N={N}  modo={MODE}  K_MAX={K_MAX}")
k2, m2 = bmc(stable, K_MAX)
report(k2, m2)

N=17  modo=SYNC  K_MAX=12
  CONTRA-EXEMPLO com k = 2 passos (validacao cruzada: OK)
  celulas iniciais vivas: [(0, 0), (0, 5), (0, 6), (5, 0), (6, 0)]
  t=0
   ###..............
   ###..............
   ###..............
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
  t=1
   ..##.............
   ...#.............
   #.#..............
   ##...............
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
   .................
  t=2
   ..##.............
   .#.#.............
   #.#..............
   ##...............
   .................
   .........

## 9. Validacao independente

Simulador sem usar a biblioteca z3 que reproduz os contra-exemplos conhecidos:

- SYNC, celulas iniciais todas mortas: o estado estabiliza (P2 violada, P1 nao).
- SYNC, so `(0,4)` viva: chega-se ao estado de erro (P1 violada).
- ASYNC, celulas iniciais todas mortas: sequencia de 9 transicoes ate ao estado de erro.

In [12]:
def make_grid(alive_boundary=()):
    g = [[0] * (N + 2) for _ in range(N + 2)]
    for c in alive_boundary:
        assert is_initial(*c)
        g[c[0]][c[1]] = 1
    for i in range(1, 4):
        for j in range(1, 4):
            g[i][j] = 1
    return g

def nxt_cell(g, i, j):
    n = sum(g[a][b] for a in (i-1, i, i+1) for b in (j-1, j, j+1) if (a, b) != (i, j))
    return 1 if (n == 3 or (g[i][j] and n == 2)) else 0

def sync_step(g):
    h = [r[:] for r in g]
    for (i, j) in NORMAL:
        h[i][j] = nxt_cell(g, i, j)
    return h

def is_error(g):
    return not any(g[i][j] for (i, j) in NORMAL)

def run_sync(alive_boundary=(), steps=200):
    g, seen = make_grid(alive_boundary), {}
    for t in range(steps):
        if is_error(g):
            return ("erro", t)
        key = tuple(map(tuple, g))
        if key in seen:
            return ("ciclo", seen[key], t - seen[key], sum(g[i][j] for (i, j) in NORMAL))
        seen[key] = t
        g = sync_step(g)
    return ("?", steps)

print("SYNC, iniciais mortas       :", run_sync())        # ('ciclo', t0, periodo, nº vivas)
print("SYNC, so (0,4) viva         :", run_sync([(0, 4)]))  # ('erro', 7)

g = make_grid()
seq = [(1,2),(2,1),(1,1),(2,3),(2,2),(1,3),(3,1),(3,2),(3,3)]
for (i, j) in seq:
    g[i][j] = nxt_cell(g, i, j)
print("ASYNC, sequencia de 9 passos -> estado de erro:", is_error(g))

SYNC, iniciais mortas       : ('ciclo', 4, 1, 7)
SYNC, so (0,4) viva         : ('erro', 7)
ASYNC, sequencia de 9 passos -> estado de erro: True
